In [3]:
import pandas as pd

df_2017 = pd.read_csv("../data/raw/amazon_india_2017.csv")

df_2017_clean = df_2017.copy()


In [4]:
print("Dataset shape:", df_2017.shape)

print("\nColumn names:")
print(df_2017.columns.tolist())

print("\nData types:")
print(df_2017.dtypes)

print("\nMissing values:")
print(df_2017.isna().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df_2017.duplicated().sum())

print("\nUnique transaction IDs:", df_2017["transaction_id"].nunique()) 

Dataset shape: (77385, 34)

Column names:
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating']

Data types:
transaction_id                str
order_date                    str
customer_id                   str
product_id                    str
product_name                  str
category                      str
subcategory                   str
brand                         str
original_price_inr            str
discount_percent          float64
discoun

In [8]:
missing_delivery = df_2017["delivery_charges"].isna()
print(
    "Missing delivery charges where final amount equals subtotal",
    (
    df_2017.loc[missing_delivery,"final_amount_inr"]
    ==df_2017.loc[missing_delivery,"subtotal_inr"]
    ).sum()
)
print("Total missing delivery charges:", missing_delivery.sum())

Missing delivery charges where final amount equals subtotal 6189
Total missing delivery charges: 6189


In [11]:
missing_delivery = df_2017["delivery_charges"].isna().sum()

In [12]:
df_2017["delivery_charges"].isna().sum()

np.int64(6189)

In [13]:
print(
    pd.crosstab(
        df_2017["is_festival_sale"],
        df_2017["festival_name"].isna()
    )
)

festival_name     False  True 
is_festival_sale              
0                     0   1573
1                   763      0
FALSE                 0   1534
False                 0  48121
No                    0   1603
TRUE                753      0
True              22317      0
Yes                 721      0


In [14]:
festival_mapping = {
    "1": True,
    "TRUE": True,
    "True": True,
    "Yes": True,
    "0": False,
    "FALSE": False,
    "False": False,
    "No": False
}

df_2017_clean["is_festival_sale"] = (
    df_2017_clean["is_festival_sale"]
    .astype(str)
    .str.strip()
    .map(festival_mapping)
)

In [15]:
df_2017_clean["is_festival_sale"].value_counts(dropna=False)

is_festival_sale
False    52831
True     24554
Name: count, dtype: int64

In [17]:
df_2017_clean["festival_name"] =(
    df_2017_clean["festival_name"].fillna("Not Applicable")
)

In [18]:
df_2017_clean["festival_name"].isna().sum()

np.int64(0)

In [19]:
df_2017_clean["delivery_charges"] = (
    df_2017_clean["delivery_charges"].fillna(0)
)

In [20]:
df_2017_clean["delivery_charges"].isna().sum()

np.int64(0)

In [22]:
df_2017_clean["customer_age_group"].value_counts(dropna=False)

customer_age_group
26-35    23791
18-25    21740
36-45    13771
NaN       9273
46-55     6636
55+       2174
Name: count, dtype: int64

In [23]:
df_2017_clean["customer_age_group"]=(
    df_2017_clean["customer_age_group"].fillna("Unknown")
)

In [24]:
df_2017_clean["customer_age_group"].isna().sum()

np.int64(0)

In [25]:
df_2017_clean["customer_age_group"].value_counts(dropna=False)

customer_age_group
26-35      23791
18-25      21740
36-45      13771
Unknown     9273
46-55       6636
55+         2174
Name: count, dtype: int64

In [26]:
df_2017["customer_rating"].value_counts(dropna=False).head(20)

customer_rating
NaN          23592
4.5          16823
5.0          12873
4.0          11954
3.5           5162
3.0           2674
4/5            776
4.5 stars      586
5.0 stars      449
4.0 stars      407
5/5            318
4              276
4.5/5.0        264
5.0/5.0        226
4.0/5.0        219
3/5            202
3.5 stars      182
5              111
3.5/5.0         96
3.0 stars       91
Name: count, dtype: int64

In [28]:
df_2017_clean["customer_rating"] = (
    df_2017_clean["customer_rating"]
    .astype("string")
    .str.replace(" stars","",regex=False)
    .str.split("/")
    .str[0]
)
df_2017_clean["customer_rating"] =pd.to_numeric (
    df_2017_clean["customer_rating"],
    errors = "coerce"

)

In [29]:
df_2017_clean["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    23592
4.5    17673
5.0    13977
4.0    13632
3.5     5440
3.0     3071
Name: count, dtype: int64

In [30]:
print("Data type:", df_2017_clean["customer_rating"].dtype)

print("\nMissing ratings:")
print(df_2017_clean["customer_rating"].isna().sum())

print("\nRating distribution:")
print(df_2017_clean["customer_rating"].value_counts().sort_index())

print("\nMinimum rating:", df_2017_clean["customer_rating"].min())
print("Maximum rating:", df_2017_clean["customer_rating"].max())

Data type: float64

Missing ratings:
23592

Rating distribution:
customer_rating
3.0     3071
3.5     5440
4.0    13632
4.5    17673
5.0    13977
Name: count, dtype: int64

Minimum rating: 3.0
Maximum rating: 5.0


In [31]:
df_2017["original_price_inr"].dtype

<StringDtype(storage='python', na_value=nan)>

In [32]:
df_2017["original_price_inr"].sample(20,random_state=42).tolist()

['99394.79',
 '42786.13',
 '93136.44',
 '44128.17',
 '25596.32',
 '52356.86',
 '110000.79',
 '107,769.2',
 '27136.77',
 '49135.38',
 '46908.39',
 '99885.08',
 '26,967.64',
 'Rs 33,691',
 '26170.89',
 '110295.57',
 '36684.78',
 '105339.38',
 '128668.67',
 '170401.43']

In [34]:
price_numeric = pd.to_numeric(
    df_2017["original_price_inr"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

(price_numeric < 0).sum()

np.int64(186)

In [35]:
df_2017.loc[
    price_numeric<0,
    "original_price_inr"
    ].value_counts().head(20)

original_price_inr
-68484.01     4
-86818.37     3
-39112.96     3
-47666.01     3
-23411.32     3
-45408.65     2
-26296.83     2
-43533.35     2
-45593.28     2
-36727.1      2
-23291.0      2
-175891.74    2
-84621.93     2
-36350.28     2
-45549.19     2
-97549.43     2
-131021.39    2
-99799.87     2
-52647.14     2
-47052.18     2
Name: count, dtype: int64

In [36]:
negative_price_rows = df_2017.loc[
    price_numeric < 0,
    ["original_price_inr", "discounted_price_inr", "discount_percent"]
]

negative_price_rows.head(10)

,original_price_inr,discounted_price_inr,discount_percent
386,-30339.6,28260.99,6.85
458,-127000.03,109748.81,13.58
1059,-43771.42,43771.42,0.00
1264,-17974.52,17974.52,0.00
1364,-36282.51,36282.51,0.00
1453,-45408.65,45408.65,0.00
2263,-21854.05,21854.05,0.00
4020,-26296.83,20838.82,20.76
4207,-41498.89,30746.85,25.91
4941,-29914.07,29914.07,0.00


In [37]:
expected_discounted_price = (
    price_numeric.abs()
    * (1 - df_2017["discount_percent"] / 100)
).round(2)

(
    expected_discounted_price.loc[price_numeric < 0]
    == df_2017.loc[
        price_numeric < 0, "discounted_price_inr"
    ].round(2)
).sum()

np.int64(84)

In [38]:
negative_mask = price_numeric < 0

discount_mismatch = (
    expected_discounted_price.round(2)
    != df_2017["discounted_price_inr"].round(2)
)

negative_mismatch_rows = df_2017.loc[
    negative_mask & discount_mismatch,
    [
        "original_price_inr",
        "discounted_price_inr",
        "discount_percent"
    ]
]

print("Mismatched rows:", len(negative_mismatch_rows))

negative_mismatch_rows.head(15)

Mismatched rows: 102


,original_price_inr,discounted_price_inr,discount_percent
386,-30339.6,28260.99,6.85
458,-127000.03,109748.81,13.58
4020,-26296.83,20838.82,20.76
4207,-41498.89,30746.85,25.91
5341,-54473.54,49743.08,8.68
5479,-49421.61,36479.64,26.19
5634,-43533.35,31229.17,28.26
5976,-32959.74,23648.57,28.25
6888,-30314.01,24504.35,19.16
8064,-169822.61,128409.71,24.39


In [40]:
negative_mismatch_rows = negative_mismatch_rows.copy()

negative_mismatch_rows["absolute_original_price"] = (
    pd.to_numeric(
        negative_mismatch_rows["original_price_inr"],
        errors="coerce"
    ).abs()
)

negative_mismatch_rows["price_difference"] = (
    negative_mismatch_rows["absolute_original_price"]
    - negative_mismatch_rows["discounted_price_inr"]
)

negative_mismatch_rows.head(15)

,original_price_inr,discounted_price_inr,discount_percent,absolute_original_price,price_difference
386,-30339.6,28260.99,6.85,30339.60,2078.61
458,-127000.03,109748.81,13.58,127000.03,17251.22
4020,-26296.83,20838.82,20.76,26296.83,5458.01
4207,-41498.89,30746.85,25.91,41498.89,10752.04
5341,-54473.54,49743.08,8.68,54473.54,4730.46
5479,-49421.61,36479.64,26.19,49421.61,12941.97
5634,-43533.35,31229.17,28.26,43533.35,12304.18
5976,-32959.74,23648.57,28.25,32959.74,9311.17
6888,-30314.01,24504.35,19.16,30314.01,5809.66
8064,-169822.61,128409.71,24.39,169822.61,41412.90


In [41]:
negative_mismatch_rows["price_difference"].describe()

count       102.000000
mean      19664.608333
std       19067.292072
min         548.130000
25%        6513.742500
50%       12972.930000
75%       24445.472500
max      102320.620000
Name: price_difference, dtype: float64

In [42]:
df_2017_clean["original_price_inr"] = price_numeric.abs()

In [43]:
print("Missing prices:", df_2017_clean["original_price_inr"].isna().sum())

print("Negative prices:", (df_2017_clean["original_price_inr"] < 0).sum())

Missing prices: 0
Negative prices: 0


In [44]:
scale_issue = (
    (df_2017_clean["original_price_inr"] / 100).round(2)
    == df_2017_clean["discounted_price_inr"].round(2)
)

print("Potential price-scale errors:", scale_issue.sum())

Potential price-scale errors: 82


In [45]:
df_2017_clean.loc[scale_issue, "original_price_inr"] = (
    df_2017_clean.loc[scale_issue, "original_price_inr"] / 100
)

In [46]:
scale_check = (
    (df_2017_clean["original_price_inr"] / 100).round(2)
    == df_2017_clean["discounted_price_inr"].round(2)
)

print("Remaining exact scale matches:", scale_check.sum())

Remaining exact scale matches: 0


In [47]:
expected_discounted_price = (
    df_2017_clean["original_price_inr"]
    * (1 - df_2017_clean["discount_percent"] / 100)
).round(2)

discount_mismatch = (
    expected_discounted_price
    != df_2017_clean["discounted_price_inr"].round(2)
)

print("Discount calculation mismatches:", discount_mismatch.sum())
print(
    "Mismatch percentage:",
    round(discount_mismatch.mean() * 100, 2),
    "%"
)

Discount calculation mismatches: 44030
Mismatch percentage: 56.9 %


In [48]:
df_2017_clean.loc[
    discount_mismatch,
    [
        "original_price_inr",
        "discount_percent",
        "discounted_price_inr",
        "quantity",
        "subtotal_inr",
        "final_amount_inr"
    ]
].head(10)

,original_price_inr,discount_percent,discounted_price_inr,quantity,subtotal_inr,final_amount_inr
1,16997.28,9.98,15301.00,1,15301.00,15301.00
8,31992.51,66.90,10588.02,2,21176.04,21176.04
12,172043.77,53.59,79851.12,1,79851.12,79851.12
14,52356.86,7.73,48309.74,1,48309.74,48309.74
17,52647.00,13.35,45620.01,1,45620.01,45620.01
18,14094.89,25.62,10484.05,1,10484.05,10484.05
21,24675.15,30.24,17212.16,1,17212.16,17212.16
26,78879.13,26.97,57603.14,1,57603.14,57603.14
29,35962.66,39.75,21667.92,2,43335.85,43335.85
30,33569.62,15.22,28461.63,2,56923.26,56923.26


In [49]:
expected_subtotal = (
    df_2017_clean["discounted_price_inr"]
    * df_2017_clean["quantity"]
).round(2)

subtotal_mismatch = (
    expected_subtotal
    != df_2017_clean["subtotal_inr"].round(2)
)

print("Subtotal mismatches:", subtotal_mismatch.sum())
print(
    "Mismatch percentage:",
    round(subtotal_mismatch.mean() * 100, 2),
    "%"
)

Subtotal mismatches: 4635
Mismatch percentage: 5.99 %


In [50]:
expected_final_amount = (
    df_2017_clean["subtotal_inr"]
    + df_2017_clean["delivery_charges"]
).round(2)

final_mismatch = (
    expected_final_amount
    != df_2017_clean["final_amount_inr"].round(2)
)

print("Final amount mismatches:", final_mismatch.sum())

print(
    "Mismatch percentage:",
    round(final_mismatch.mean() * 100, 2),
    "%"
)

Final amount mismatches: 0
Mismatch percentage: 0.0 %


In [51]:
print("Order date data type:", df_2017["order_date"].dtype)

print("\nSample order dates:")
print(df_2017["order_date"].sample(10, random_state=42).tolist())

Order date data type: str

Sample order dates:
['2017-10-31', '2017-10-24', '2017-11-03', '15/10/2017', '2017-06-12', '06/30/2017', '2017-04-01', '2017-10-28', '2017-10-02', '2017-02-23']


In [52]:
date_text = df_2017["order_date"].str.strip()

parsed_dates = pd.Series(
    pd.NaT,
    index=df_2017.index,
    dtype="datetime64[ns]"
)

# Format 1: YYYY-MM-DD
mask_ymd = date_text.str.match(r"^\d{4}-\d{1,2}-\d{1,2}$")

parsed_dates.loc[mask_ymd] = pd.to_datetime(
    date_text.loc[mask_ymd],
    format="%Y-%m-%d",
    errors="coerce"
)

# Format 2: DD/MM/YYYY
mask_dmy = date_text.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$")

parsed_dates.loc[mask_dmy] = pd.to_datetime(
    date_text.loc[mask_dmy],
    format="%d/%m/%Y",
    errors="coerce"
)

# Format 3: MM/DD/YYYY
mask_mdy = (
    parsed_dates.isna()
    & date_text.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$")
)

parsed_dates.loc[mask_mdy] = pd.to_datetime(
    date_text.loc[mask_mdy],
    format="%m/%d/%Y",
    errors="coerce"
)

# Format 4: DD-MM-YYYY
mask_dmy_dash = date_text.str.match(r"^\d{1,2}-\d{1,2}-\d{4}$")

parsed_dates.loc[mask_dmy_dash] = pd.to_datetime(
    date_text.loc[mask_dmy_dash],
    format="%d-%m-%Y",
    errors="coerce"
)

df_2017_clean["order_date"] = parsed_dates

In [53]:
print("Missing dates after parsing:",
      df_2017_clean["order_date"].isna().sum())

Missing dates after parsing: 0


In [54]:
month_mismatch = (
    df_2017_clean["order_date"].dt.month
    != df_2017_clean["order_month"]
)

year_mismatch = (
    df_2017_clean["order_date"].dt.year
    != df_2017_clean["order_year"]
)

print("Month mismatches:", month_mismatch.sum())
print("Year mismatches:", year_mismatch.sum())

Month mismatches: 678
Year mismatches: 0


In [55]:
month_mismatch_rows = df_2017.loc[
    month_mismatch,
    ["order_date", "order_month", "order_year"]
].copy()

month_mismatch_rows["parsed_date"] = (
    df_2017_clean.loc[month_mismatch, "order_date"]
)

month_mismatch_rows.head(20)

,order_date,order_month,order_year,parsed_date
125,01/06/2017,1,2017,2017-06-01
241,01/12/2017,1,2017,2017-12-01
284,01/05/2017,1,2017,2017-05-01
349,01/04/2017,1,2017,2017-04-01
393,01/11/2017,1,2017,2017-11-01
431,01/08/2017,1,2017,2017-08-01
482,01/07/2017,1,2017,2017-07-01
591,01/05/2017,1,2017,2017-05-01
667,01/03/2017,1,2017,2017-03-01
682,01/06/2017,1,2017,2017-06-01


In [56]:
# Find rows where the parsed month doesn't match order_month
month_mismatch = (
    df_2017_clean["order_date"].dt.month
    != df_2017_clean["order_month"]
)

# Get the original date strings for those rows
original_dates = df_2017.loc[month_mismatch, "order_date"].str.strip()

# Re-parse those dates as MM/DD/YYYY
corrected_dates = pd.to_datetime(
    original_dates,
    format="%m/%d/%Y",
    errors="coerce"
)

# Update the cleaned dates
df_2017_clean.loc[month_mismatch, "order_date"] = corrected_dates

In [57]:
month_mismatch = (
    df_2017_clean["order_date"].dt.month
    != df_2017_clean["order_month"]
)

year_mismatch = (
    df_2017_clean["order_date"].dt.year
    != df_2017_clean["order_year"]
)

print("Remaining month mismatches:", month_mismatch.sum())
print("Remaining year mismatches:", year_mismatch.sum())
print("Missing parsed dates:", df_2017_clean["order_date"].isna().sum())

Remaining month mismatches: 0
Remaining year mismatches: 0
Missing parsed dates: 0


In [58]:
expected_quarter = df_2017_clean["order_date"].dt.quarter

quarter_mismatch = (
    expected_quarter != df_2017_clean["order_quarter"]
)

print("Quarter mismatches:", quarter_mismatch.sum())

Quarter mismatches: 0


In [59]:
categorical_columns = [
    "category",
    "subcategory",
    "brand",
    "payment_method",
    "delivery_type",
    "return_status",
    "customer_tier",
    "customer_spending_tier",
    "customer_city",
    "customer_state"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df_2017_clean[column].value_counts().head(15))


--- category ---
category
Electronics                  77305
Electronicss                    28
ELECTRONICS                     25
Electronic                      16
Electronics & Accessories       11
Name: count, dtype: int64

--- subcategory ---
subcategory
Smartphones           54087
Laptops                7312
Tablets                6745
Smart Watch            4899
Audio                  3018
TV & Entertainment     1324
Name: count, dtype: int64

--- brand ---
brand
Samsung      17096
OnePlus      11571
Xiaomi       11300
Apple         9058
Motorola      4734
Oppo          4338
Vivo          4120
Lenovo        1745
ASUS          1415
Acer          1304
Fitbit        1113
Alienware     1017
Realme         995
Dell           857
Noise          769
Name: count, dtype: int64

--- payment_method ---
payment_method
COD            46386
Credit Card    12419
Debit Card      9258
UPI             5471
Net Banking     3851
Name: count, dtype: int64

--- delivery_type ---
delivery_type
Standa

In [60]:
df_2017_clean["category"] = (
    df_2017_clean["category"]
    .replace({
        "Electronicss": "Electronics",
        "ELECTRONICS": "Electronics",
        "Electronic": "Electronics"
    })
)

In [61]:
print(df_2017_clean["category"].value_counts())

category
Electronics                  77374
Electronics & Accessories       11
Name: count, dtype: int64


In [62]:
print(sorted(df_2017_clean["customer_city"].dropna().unique()))

['Ahmedabad', 'Aligarh', 'Allahabad', 'BANGALORE', 'Bangalore', 'Banglore', 'Bareilly', 'Bengalore', 'Bengaluru', 'Bhubaneswar', 'Bombay', 'CHENNAI', 'Calcutta', 'Chandigarh', 'Chennai', 'Chennai ', 'Coimbatore', 'DELHI', 'Delhi', 'Delhi NCR', 'Gorakhpur', 'Hyderabad', 'Indore', 'Jaipur', 'KOLKATA', 'Kanpur', 'Kochi', 'Kolkata', 'Kolkata ', 'Lucknow', 'Ludhiana', 'MUMBAI', 'Madras', 'Meerut', 'Moradabad', 'Mumbai', 'Mumbai ', 'Nagpur', 'New Delhi', 'Patna', 'Pune', 'Saharanpur', 'Surat', 'Vadodara', 'Varanasi', 'Visakhapatnam', 'chenai', 'delhi', 'kolkata', 'mumba']


In [63]:
city_mapping = {
    "BANGALORE": "Bengaluru",
    "Bangalore": "Bengaluru",
    "Banglore": "Bengaluru",
    "Bengalore": "Bengaluru",
    "CHENNAI": "Chennai",
    "Chennai ": "Chennai",
    "chenai": "Chennai",
    "Madras": "Chennai",
    "DELHI": "Delhi",
    "delhi": "Delhi",
    "KOLKATA": "Kolkata",
    "kolkata": "Kolkata",
    "Kolkata ": "Kolkata",
    "Calcutta": "Kolkata",
    "MUMBAI": "Mumbai",
    "Mumbai ": "Mumbai",
    "mumba": "Mumbai"
}

df_2017_clean["customer_city"] = (
    df_2017_clean["customer_city"]
    .replace(city_mapping)
    .str.strip()
)

In [64]:
print(sorted(df_2017_clean["customer_city"].dropna().unique()))

['Ahmedabad', 'Aligarh', 'Allahabad', 'Bareilly', 'Bengaluru', 'Bhubaneswar', 'Bombay', 'Chandigarh', 'Chennai', 'Coimbatore', 'Delhi', 'Delhi NCR', 'Gorakhpur', 'Hyderabad', 'Indore', 'Jaipur', 'Kanpur', 'Kochi', 'Kolkata', 'Lucknow', 'Ludhiana', 'Meerut', 'Moradabad', 'Mumbai', 'Nagpur', 'New Delhi', 'Patna', 'Pune', 'Saharanpur', 'Surat', 'Vadodara', 'Varanasi', 'Visakhapatnam']


In [65]:
print(sorted(df_2017_clean["customer_state"].dropna().unique()))

['Andhra Pradesh', 'Bihar', 'Delhi', 'Gujarat', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Odisha', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Telangana', 'Uttar Pradesh', 'West Bengal']


In [67]:
print("Total missing values:", df_2017_clean.isnull().sum())

Total missing values: transaction_id                0
order_date                    0
customer_id                   0
product_id                    0
product_name                  0
category                      0
subcategory                   0
brand                         0
original_price_inr            0
discount_percent              0
discounted_price_inr          0
quantity                      0
subtotal_inr                  0
delivery_charges              0
final_amount_inr              0
customer_city                 0
customer_state                0
customer_tier                 0
customer_spending_tier        0
customer_age_group            0
payment_method                0
delivery_days                 0
delivery_type                 0
is_prime_member               0
is_festival_sale              0
festival_name                 0
customer_rating           23592
return_status                 0
order_month                   0
order_year                    0
order_quarter     

In [68]:
# 1. Check missing values
print("Total missing values:", df_2017_clean.isnull().sum().sum())

# 2. Check duplicate rows
print("Duplicate rows:", df_2017_clean.duplicated().sum())

# 3. Check duplicate transaction IDs
print("Duplicate transaction IDs:",
      df_2017_clean["transaction_id"].duplicated().sum())

# 4. Check negative prices
print("Negative original prices:",
      (df_2017_clean["original_price_inr"] < 0).sum())

# 5. Check date consistency
print("Month mismatches:",
      (df_2017_clean["order_date"].dt.month
       != df_2017_clean["order_month"]).sum())

print("Year mismatches:",
      (df_2017_clean["order_date"].dt.year
       != df_2017_clean["order_year"]).sum())

print("Quarter mismatches:",
      (df_2017_clean["order_date"].dt.quarter
       != df_2017_clean["order_quarter"]).sum())

Total missing values: 23592
Duplicate rows: 0
Duplicate transaction IDs: 0
Negative original prices: 0
Month mismatches: 0
Year mismatches: 0
Quarter mismatches: 0


In [69]:
df_2017_clean.isnull().sum()[df_2017_clean.isnull().sum() > 0]

customer_rating    23592
dtype: int64

In [70]:
df_2017_clean.to_csv(
    "../data/cleaned/amazon_india_2017_clean.csv",
    index=False
)